# Pathway Subtyping Framework: 47-Dataset Benchmark Calibration
## Full Pipeline (GEO datasets)
## Run on Google Colab Pro/Pro+ (100GB+ RAM)

**Pipeline:**
1. Load datasets
2. Score pathways using ssGSEA
3. Run clustering on pathway scores
4. Compute bootstrap stability metrics

**Estimated Runtime:** 2-3 hours for 35 GEO datasets

In [ ]:
import os
import sys

os.environ['GEOPARSE_USE_HTTP_FOR_FTP'] = 'yes'

!pip install -q GEOparse scanpy numpy 'pandas==2.2.2' scikit-learn scipy -U
!pip install -q pathway-subtyping

print('OK: Dependencies installed')

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/gdrive')
output_dir = '/content/gdrive/My Drive/Colab Notebooks/pathway-benchmark-results'
os.makedirs(output_dir, exist_ok=True)

print(f'OK: Google Drive mounted to {output_dir}')

In [ ]:
repo_url = 'https://codeberg.org/pathways/pathway-subtyping-framework.git'
repo_path = '/content/pathway-subtyping-framework'
branch = 'feat/benchmark-calibration-47'

if os.path.exists(repo_path):
    !rm -rf {repo_path}

!git clone -b {branch} {repo_url} {repo_path}
sys.path.insert(0, f'{repo_path}/scripts')

print(f'OK: Repository cloned')

In [ ]:
import logging
import pandas as pd
import numpy as np
import warnings
import time
from typing import Tuple, Dict

from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
    run_clustering,
    select_n_clusters,
    ClusteringAlgorithm
)
from benchmark_data_loaders import BenchmarkDataLoaderFactory
from benchmark_validation import compute_silhouette, bootstrap_ari_stability

warnings.filterwarnings('ignore', message='.*Columns.*have mixed types.*')
logging.basicConfig(level=logging.INFO, format='[%(name)s] %(message)s')
logger = logging.getLogger('BENCHMARK')

print('OK: Modules imported')

In [ ]:
# Load pathways from GMT file
def load_pathways_from_gmt(gmt_path):
    pathways = {}
    with open(gmt_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            fields = line.split('\t')
            pathway_name = fields[0]
            genes = fields[2:]
            pathways[pathway_name] = genes
    return pathways

# Load autism pathways
gmt_file = Path(repo_path) / 'data' / 'pathways' / 'autism_pathways.gmt'
pathways = load_pathways_from_gmt(gmt_file)
print(f'OK: Loaded {len(pathways)} pathways from {gmt_file.name}')

# Load manifest
manifest_path = Path(repo_path) / 'data' / 'benchmarks' / 'benchmark_47datasets_manifest.csv'
manifest_df = pd.read_csv(manifest_path)
datasets_df = manifest_df[manifest_df['source'] == 'GEO'].reset_index(drop=True)
total_datasets = len(datasets_df)

print(f'OK: Loaded manifest with {total_datasets} GEO datasets')
print(f'\nDatasets to process:')
for i, row in datasets_df.iterrows():
    print(f"  [{i+1:2d}] {row['dataset_name'][:30]:30s} ({row['domain']})")

In [ ]:
# Configuration
BOOTSTRAP_ITERATIONS = 10
K_RANGE = (2, 7)
RANDOM_SEED = 42
output_csv = Path(output_dir) / 'bootstrap_threshold_calibration_35geo_datasets.csv'

def run_full_pipeline(
    gene_expression: np.ndarray,
    true_labels: np.ndarray,
    pathways: dict,
    seed: int = 42,
    k_range: Tuple[int, int] = (2, 7),
    bootstrap_iterations: int = 10
) -> Dict:
    try:
        # Step 1: Score pathways
        logger.info(f'      - Scoring {len(pathways)} pathways...')
        scoring_result = score_pathways_from_expression(
            gene_expression=gene_expression,
            pathways=pathways,
            method=ExpressionScoringMethod.SSGSEA,
            min_genes_per_pathway=2,
            seed=seed,
            show_progress=False
        )
        pathway_scores = scoring_result.pathway_scores.values
        n_pathways_scored = scoring_result.n_pathways_scored
        logger.info(f'      - Scored {n_pathways_scored} pathways')
        
        # Step 2: Select k
        logger.info(f'      - Selecting k...')
        k_list = list(range(k_range[0], k_range[1] + 1))
        k_selection = select_n_clusters(data=pathway_scores, k_range=k_list, method='bic', seed=seed)
        optimal_k = k_selection.optimal_k
        logger.info(f'      - Optimal k: {optimal_k}')
        
        # Step 3: Run clustering
        logger.info(f'      - Clustering with GMM...')
        clustering_result = run_clustering(
            data=pathway_scores,
            n_clusters=optimal_k,
            algorithm=ClusteringAlgorithm.GMM,
            seed=seed
        )
        clusters = clustering_result.labels
        
        # Step 4: Silhouette
        logger.info(f'      - Computing silhouette...')
        silhouette = compute_silhouette(pathway_scores, clusters)
        
        # Step 5: Bootstrap
        def cluster_func(data):
            k_sel = select_n_clusters(data=data, k_range=k_list, method='bic', seed=seed)
            result = run_clustering(data=data, n_clusters=k_sel.optimal_k, algorithm=ClusteringAlgorithm.GMM, seed=seed)
            return result.labels
        
        logger.info(f'      - Bootstrap ({bootstrap_iterations} replicates)...')
        bootstrap_ari = bootstrap_ari_stability(
            pathway_scores, true_labels, cluster_func,
            n_iterations=bootstrap_iterations, sample_fraction=0.8, seed=seed
        )
        bootstrap_ari_5th = np.percentile(bootstrap_ari, 5)
        
        n_detected = len(np.unique(clusters))
        n_true = len(np.unique(true_labels))
        n_samples = gene_expression.shape[0]
        
        logger.info(f'      OK: Silhouette={silhouette:.4f}, Bootstrap ARI(5%)={bootstrap_ari_5th:.4f}')
        
        return {
            'silhouette': float(silhouette),
            'bootstrap_ari_5th_percentile': float(bootstrap_ari_5th),
            'n_samples': int(n_samples),
            'n_detected_clusters': int(n_detected),
            'n_true_clusters': int(n_true),
            'n_pathways_scored': int(n_pathways_scored),
            'status': 'PASS'
        }
    except Exception as e:
        logger.error(f'      ERROR: {e}')
        return {'status': 'ERROR', 'error_message': str(e)}

print('OK: Pipeline function defined')

In [ ]:
logger.info('='*80)
logger.info(f'RUNNING BENCHMARK ON {total_datasets} GEO DATASETS')
logger.info('='*80)

results = []
success_count = 0
error_count = 0
start_time = time.time()

for idx, row in datasets_df.iterrows():
    dataset_num = idx + 1
    dataset_id = row['dataset_id']
    dataset_name = row['dataset_name']
    domain = row['domain']
    
    logger.info(f'\n[{dataset_num:2d}/{total_datasets}] {dataset_name} ({domain})')
    logger.info('-'*70)
    
    try:
        logger.info(f'  Loading...')
        expression, labels = BenchmarkDataLoaderFactory.load(row)
        logger.info(f'  Loaded: {expression.shape[0]} samples × {expression.shape[1]} genes')
        
        pipeline_result = run_full_pipeline(
            gene_expression=expression,
            true_labels=labels,
            pathways=pathways,
            seed=RANDOM_SEED,
            k_range=K_RANGE,
            bootstrap_iterations=BOOTSTRAP_ITERATIONS
        )
        
        result = {
            'dataset_id': dataset_id,
            'dataset_name': dataset_name,
            'domain': domain,
            'source': 'GEO',
            'silhouette': pipeline_result.get('silhouette'),
            'bootstrap_ari_5th_percentile': pipeline_result.get('bootstrap_ari_5th_percentile'),
            'n_samples': pipeline_result.get('n_samples'),
            'n_detected_clusters': pipeline_result.get('n_detected_clusters'),
            'n_true_clusters': pipeline_result.get('n_true_clusters'),
            'n_pathways_scored': pipeline_result.get('n_pathways_scored'),
            'status': pipeline_result.get('status', 'UNKNOWN'),
            'error_message': pipeline_result.get('error_message', '')
        }
        
        results.append(result)
        if pipeline_result.get('status') == 'PASS':
            success_count += 1
        else:
            error_count += 1
        
        if dataset_num % 5 == 0 or dataset_num == total_datasets:
            results_df = pd.DataFrame(results)
            results_df.to_csv(output_csv, index=False)
            elapsed = time.time() - start_time
            logger.info(f'  [Checkpoint] {dataset_num}/{total_datasets} in {elapsed/60:.1f} min')
    
    except Exception as e:
        logger.error(f'  FATAL ERROR: {e}')
        result = {
            'dataset_id': dataset_id,
            'dataset_name': dataset_name,
            'domain': domain,
            'source': 'GEO',
            'status': 'ERROR',
            'error_message': str(e)
        }
        results.append(result)
        error_count += 1

logger.info('\n' + '='*80)
logger.info(f'SUMMARY: {success_count}/{len(results)} datasets passed')
logger.info('='*80)

results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)
logger.info(f'Results saved: {output_csv}')

if success_count > 0:
    passed_df = results_df[results_df['status'] == 'PASS']
    logger.info(f'\nSilhouette - Mean: {passed_df["silhouette"].mean():.4f}, Range: [{passed_df["silhouette"].min():.4f}, {passed_df["silhouette"].max():.4f}]')
    logger.info(f'Bootstrap ARI 5% - Mean: {passed_df["bootstrap_ari_5th_percentile"].mean():.4f}, Range: [{passed_df["bootstrap_ari_5th_percentile"].min():.4f}, {passed_df["bootstrap_ari_5th_percentile"].max():.4f}]')

elapsed_total = time.time() - start_time
logger.info(f'\nTotal runtime: {elapsed_total/3600:.2f} hours')
logger.info('OK: BENCHMARK COMPLETE')

In [ ]:
results_df = pd.read_csv(output_csv)
print('\nFINAL RESULTS')
print('='*80)
print(f'Datasets: {len(results_df)}')
print(f'Passed: {(results_df["status"] == "PASS").sum()}')
print(f'Failed: {(results_df["status"] == "ERROR").sum()}')

if (results_df['status'] == 'PASS').sum() > 0:
    print('\nTop 10 by Silhouette:')
    top = results_df[results_df['status'] == 'PASS'].nlargest(10, 'silhouette')
    print(top[['dataset_name', 'domain', 'silhouette', 'bootstrap_ari_5th_percentile']].to_string())

print(f'\nResults file: {output_csv}')